In [2]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms, models
from torchvision.models import vit_b_16, ViT_B_16_Weights
from torch.utils.data import DataLoader
from pathlib import Path
import os

c:\Users\Dillon\anaconda3\envs\dl\Lib\site-packages\torchvision\io\image.py:13: UserWarning: Failed to load image Python extension: '[WinError 127] The specified procedure could not be found'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(


In [ ]:
# Define the path to the dataset (Note: 0 is real, while 1 is fake)
test_path = Path("Enter path to test dataset here")
train_path = Path("Enter path to train dataset here")


# Transforms
train_transformer = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)

test_transformer = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)
#print(f"Train path: {train_path}, Test path: {test_path}")
# Conversion of files to PyTorch Dataset and DataLoader
train_data = datasets.ImageFolder(root=train_path, transform=train_transformer)
test_data = datasets.ImageFolder(root=test_path, transform=test_transformer)

train_loader = DataLoader(train_data, batch_size=50, shuffle=True, num_workers=4)
test_loader = DataLoader(test_data, batch_size=50, shuffle=False, num_workers=4)

In [4]:
def ViTBinaryClassifier(pretrained = True, freeze_features = False, num_classes = 1):
    model = models.vit_b_16(weights=ViT_B_16_Weights.DEFAULT if pretrained else None)
    model.heads.head = nn.Sequential(
        nn.ReLU(), 
        nn.Linear(768, num_classes),  # Binary classification
    )
    
    if freeze_features:
        for param in model.parameters():
            param.requires_grad = False
        for param in model.classifier.parameters():
            param.requires_grad = True
        
    
    return model
    
model = ViTBinaryClassifier(pretrained=True, freeze_features=False, num_classes=1).to('cuda')

In [5]:
# Define loss , optimizer, and learning rate scheduler, accuracy, false pos, false neg functions
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.1, patience=3)

In [6]:
# Training Loop
train_size = 10000
acc_list = []
fp_list = []
fn_list = []
loss_list = []

test_size = 1000
test_loss_list = []
test_acc_list = []
test_fp_list = []
test_fn_list = []

epoch_list = [i for i in range(1, 101)]
EPOCHS = 5

for epoch in range(EPOCHS):
    # Model Training
    model.train()
    
    correct = 0
    fp = 0
    fn = 0
    running_loss = 0
    losses = []
    
    
    for i, inp in enumerate(train_loader):
        inputs, labels = inp
        inputs, labels = inputs.to("cuda"), labels.to("cuda")

        outputs = model(inputs).squeeze()
        preds = torch.round(torch.sigmoid(outputs))
        
        
        loss = loss_fn(outputs, labels.type(torch.float))
        losses.append(loss.item())
        correct += (preds.view(-1) == labels).sum().item()
        fp += ((preds.view(-1) == 1) & (labels == 0)).sum().item()
        fn += ((preds.view(-1) == 0) & (labels == 1)).sum().item()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        print(f"Batch {i + 1}, Loss: {loss.item()}")

    acc_list.append(correct / train_size * 100)
    fp_list.append(fp / train_size  * 100)
    fn_list.append(fn / train_size * 100)
    avg_loss = sum(losses) / len(losses)
    loss_list.append(avg_loss)
    scheduler.step(avg_loss)
    
    # Model Evaluation
    model.eval()
    
    correct = 0
    fp = 0
    fn = 0
    running_loss = 0
    losses = []
    
    with torch.inference_mode():
        for i, inp in enumerate(test_loader):
            inputs, labels = inp
            inputs, labels = inputs.to("cuda"), labels.to("cuda")

            outputs = model(inputs).squeeze()
            preds = torch.round(torch.sigmoid(outputs))
            
            
            loss = loss_fn(outputs, labels.type(torch.float))
            losses.append(loss.item())
            correct += (preds.view(-1) == labels).sum().item()
            fp += ((preds.view(-1) == 1) & (labels == 0)).sum().item()
            fn += ((preds.view(-1) == 0) & (labels == 1)).sum().item()

            running_loss += loss.item()

    avg_loss = sum(losses) / len(losses)
    test_loss_list.append(avg_loss)
    test_acc_list.append(correct / test_size * 100)
    test_fp_list.append(fp / test_size * 100)
    test_fn_list.append(fn / test_size * 100)

    print(f"Epoch {epoch + 1}, Train Loss: {loss_list[epoch]}, Train Acc: {acc_list[epoch]}%, FP: {fp_list[epoch]}%, FN: {fn_list[epoch]}%")
    print(f"Epoch {epoch + 1}, Test Loss: {test_loss_list[epoch]}, Test Acc: {test_acc_list[epoch]}%, FP: {test_fp_list[epoch]}%, FN: {test_fn_list[epoch]}%")

    

Batch 1, Loss: 0.723149836063385
Batch 2, Loss: 0.6769161820411682
Batch 3, Loss: 0.6968643665313721
Batch 4, Loss: 0.6150311231613159
Batch 5, Loss: 0.6736277341842651
Batch 6, Loss: 0.655143141746521
Batch 7, Loss: 0.6273554563522339
Batch 8, Loss: 0.6172626614570618
Batch 9, Loss: 0.608917236328125
Batch 10, Loss: 0.6220797300338745
Batch 11, Loss: 0.5603761672973633
Batch 12, Loss: 0.5808829069137573
Batch 13, Loss: 0.5592674612998962
Batch 14, Loss: 0.5953180193901062
Batch 15, Loss: 0.6820393204689026
Batch 16, Loss: 0.5600694417953491
Batch 17, Loss: 0.7436496019363403
Batch 18, Loss: 0.6710899472236633
Batch 19, Loss: 0.6880108714103699
Batch 20, Loss: 0.6864314079284668
Batch 21, Loss: 0.6701598763465881
Batch 22, Loss: 0.67974454164505
Batch 23, Loss: 0.6750795841217041
Batch 24, Loss: 0.6747422814369202
Batch 25, Loss: 0.7026609778404236
Batch 26, Loss: 0.6960800886154175
Batch 27, Loss: 0.6406444311141968
Batch 28, Loss: 0.667344331741333
Batch 29, Loss: 0.689550518989563
B

In [7]:
import pickle
torch.save(model.state_dict(), "C:\Code\Model_Weights\ViT_Mixed.pth")